> Os exemplos desta página usam dados sintéticos gerados pelo projeto. Não são observações ASTER nem permitem interpretar valores como reflectância, temperatura ou classes reais. Veja [os dados de exemplo](../../../data/README.md).



# Capítulo 01 — Imagens, Arrays e Matrizes

> **Referência:** Canty (2019), Cap. 1 — *Images, Arrays, and Matrices*.  
> Notas: `../aulas/00_teoria.md`

Objetivo: tratar a imagem como array multibanda, revisar a matemática mínima e carregar o raster sintético para estatísticas e visualização.

## 1. Intuição

Uma cena óptica (ou SAR multibanda) não é “uma foto” só: é um **cubo** $g_k(i,j)$ — linhas × colunas × bandas.

Em cada pixel espacial $(i,j)$, empilhamos as bandas em um **vetor** $\mathbf{g}\in\mathbb{R}^N$. Classificação, PCA e detecção de mudança operam sobre esses vetores (e sobre a nuvem de pontos que eles formam).

Neste notebook usamos o raster sintético `data/exemplos/exemplo_sintetico.tif` (na raiz do repositório) e salvamos figuras em `../figuras/`.

## 2. Matemática essencial

**Matriz / array:** armazenamos tipicamente `(bandas, linhas, colunas)` em NumPy.

**Vetor de pixel:** $\mathbf{g}=(g_1,\ldots,g_N)^\top$.

**Normalização min–max** (só para visualização):

$$
g' = \frac{g - g_{\min}}{g_{\max}-g_{\min}}
$$

A matriz de covariância entre bandas e a PCA aparecem de forma rápida no Cap. 1 e são aprofundadas nos Caps. 2–3.

## 3. Implementação — carregar raster sintético

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "study_utils").is_dir())
DATA = REPO / "data/exemplos/exemplo_sintetico.tif"
OUT = REPO / "conteudo/fundamentos/figuras"
OUT.mkdir(parents=True, exist_ok=True)


def load_raster(path):
    """Lê multibanda como array (bands, rows, cols).

    Ordem: rasterio → osgeo.gdal → erro documentado.
    """
    path = Path(path)
    try:
        import rasterio

        with rasterio.open(path) as src:
            data = src.read().astype(np.float64)
            meta = {"driver": "rasterio", "crs": str(src.crs), "count": src.count}
        return data, meta
    except Exception as e_rio:
        rio_err = e_rio

    try:
        from osgeo import gdal
        from osgeo.gdalconst import GA_ReadOnly

        gdal.AllRegister()
        ds = gdal.Open(str(path), GA_ReadOnly)
        if ds is None:
            raise RuntimeError(f"GDAL retornou None para {path}")
        bands, rows, cols = ds.RasterCount, ds.RasterYSize, ds.RasterXSize
        data = np.empty((bands, rows, cols), dtype=np.float64)
        for b in range(bands):
            data[b] = ds.GetRasterBand(b + 1).ReadAsArray().astype(np.float64)
        meta = {"driver": "gdal", "projection": ds.GetProjection(), "count": bands}
        ds = None
        return data, meta
    except Exception as e_gdal:
        raise RuntimeError(
            "Falha ao ler o raster. Instale `rasterio` ou GDAL (`osgeo`). "
            f"Caminho: {path}\n"
            f"rasterio: {rio_err}\ngdal: {e_gdal}"
        )


image, meta = load_raster(DATA)
nbands, nrows, ncols = image.shape
print(f"Arquivo: {DATA.resolve()}")
print(f"Shape (bandas, linhas, colunas): {image.shape}")
print(f"Backend: {meta.get('driver')}")
print(f"dtype efetivo: {image.dtype}")

print("\nEstatísticas por banda:")
print(f"{'banda':>6} {'min':>12} {'max':>12} {'média':>12} {'std':>12}")
for b in range(nbands):
    band = image[b]
    print(
        f"{b+1:6d} {band.min():12.4f} {band.max():12.4f} "
        f"{band.mean():12.4f} {band.std():12.4f}"
    )

## 4. Visualizações

Uma banda em escala de cinza (min–max) e um grid com várias bandas. Figuras em `../figuras/`.

In [ ]:
def stretch01(band):
    mn, mx = np.nanmin(band), np.nanmax(band)
    if mx <= mn:
        return np.zeros_like(band, dtype=np.float64)
    return (band - mn) / (mx - mn)


# --- uma banda ---
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(stretch01(image[0]), cmap="gray")
ax.set_title("Exemplo sintético — banda 1 (min–max)")
ax.set_axis_off()
fig.colorbar(im, ax=ax, fraction=0.046)
fig.savefig(OUT / "banda_1.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print(f"Salvo: {OUT / 'banda_1.png'}")

# --- grid de bandas ---
n_show = min(nbands, 9)
ncols_g = 3
nrows_g = int(np.ceil(n_show / ncols_g))
fig, axes = plt.subplots(nrows_g, ncols_g, figsize=(10, 3.2 * nrows_g))
axes = np.atleast_1d(axes).ravel()
for i in range(n_show):
    axes[i].imshow(stretch01(image[i]), cmap="gray")
    axes[i].set_title(f"Banda {i+1}")
    axes[i].set_axis_off()
for j in range(n_show, len(axes)):
    axes[j].set_axis_off()
fig.suptitle("Exemplo sintético — grid de bandas", y=1.02)
fig.tight_layout()
fig.savefig(OUT / "grid_bandas.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print(f"Salvo: {OUT / 'grid_bandas.png'}")

## 5. Exercícios

Scripts em `../exercicios/` (requerem GDAL/`osgeo`):

```bash
cd ../exercicios

# Ex. 1.1 — exibe a n-ésima banda (janela interativa se o backend permitir)
python ex01_1.py ../../../data/exemplos/exemplo_sintetico.tif 1

# Ex. 1.2 — PCA e grava GeoTIFF das componentes
python ex01_2.py ../../../data/exemplos/exemplo_sintetico.tif ../figuras/aster_pca.tif
```

Compare a PCA do exercício com o tratamento do Cap. 3 (`pca.md` e `03_teoria.ipynb`).

## 6. Aplicação mestrado — Sentinel-1 Cuiabá

Já existem visualizações de referência em `../figuras/`:

- `cuiaba_sentinel1_2025_VV.png`
- `cuiaba_sentinel1_2025_VH.png`

Em SAR, VV e VH desempenham papel análogo a “bandas”: o pixel vira um vetor (possivelmente com mais polarizações ou multi-temporal).  
**Próximos passos naturais:** histograma/covariância (Cap. 2), PCA/MNF (Cap. 3), e depois filtragem/classificação com o stack VV–VH da área de estudo.

In [ ]:
# Confere se as figuras SAR de referência estão presentes
for name in ("cuiaba_sentinel1_2025_VV.png", "cuiaba_sentinel1_2025_VH.png"):
    p = OUT / name
    print(f"{'OK' if p.exists() else 'AUSENTE'}: {p}")